> # **Multi-Sensor SAVI Analysis and Time-Series Export**

> ## **Final Assessment**

> This notebook evaluates vegetation using the Soil-Adjusted Vegetation Index (SAVI) from Sentinel-2 and Landsat 8 imagery over the study area for the period 2016–2025.

In [88]:
import geopandas as gpd #To handle geospatial vector data (points, lines, polygons).
import rasterio #To read, write, and process raster data (satellite images, GeoTIFF).
import rasterio.mask #To mask or crop raster data using a geographic shape.
import rasterio.warp #To reproject and transform raster data between coordinate systems.
from rasterio.enums import Resampling #To change raster resolution using methods like bilinear or nearest neighbor.
import ee #To work with Google Earth Engine and satellite imagery.
import numpy as np #To perform numerical and array calculations efficiently.
import matplotlib.pyplot as plt #To create plots and visualize data.
from pathlib import Path #To handle file and folder paths in Python.
import geemap  # To create interactive maps and visualize Earth Engine data.

> Eestablish an authenticated session with the Google Earth Engine (GEE) Python API.

> The standard initialization call `ee.Initialize()` verifies your GEE credentials. Wrapping this step inside a `try-except` block ensures that any authentication errors or connection failures are caught gracefully rather than crashing your notebook execution.

In [89]:
ee.Authenticate()

True

In [90]:
# Initialise Google Earth Engine API
try:
    ee.Initialize()
    print("Google Earth Engine initialised successfully.")
except Exception as e:
    print(f"Error initialising GEE: {e}")

Google Earth Engine initialised successfully.


In [91]:
root_folder=Path(r"C:\Users\user\Downloads\UNAL\GEOPROCESSING")
# Load Colombian municipalities
gdf = gpd.read_file(root_folder /"municipios_colombia.gpkg")

# Display initial spatial reference system
print(f"Initial CRS: {gdf.crs}")

# Reproject to EPSG:9377
gdf_unico = gdf.to_crs(epsg=9377)
#Check the attributes table of our geopandas dataframe
gdf_unico.head()

Initial CRS: EPSG:3116


,DPTO_CCDGO,MPIO_CCDGO,MPIO_CNMBR,MPIO_CDPMP,VERSION,AREA,LATITUD,LONGITUD,STCTNENCUE,STP3_1_SI,...,STP51_PRIM,STP51_SECU,STP51_SUPE,STP51_POST,STP51_13_E,STP51_99_E,Shape_Leng,Shape_Area,Codigo_Mun,geometry
0,18,001,FLORENCIA,18001,2018,2.547638e+09,1.749139,-75.558239,71877.0,32.0,...,48848.0,59610.0,21898.0,4592.0,5892.0,3799.0,2.942508,0.206928,18001,"MULTIPOLYGON (((4730856.146 1800689.038, 47308..."
1,18,029,ALBANIA,18029,2018,4.141221e+08,1.227865,-75.882327,2825.0,24.0,...,1940.0,1712.0,231.0,41.0,215.0,46.0,1.112829,0.033618,18029,"MULTIPOLYGON (((4677933.827 1709133.846, 46779..."
2,18,094,BELÉN DE LOS ANDAQUÍES,18094,2018,1.191619e+09,1.500923,-75.875645,4243.0,54.0,...,3541.0,3340.0,490.0,119.0,720.0,123.0,2.234657,0.096745,18094,"MULTIPOLYGON (((4690015.614 1751610.86, 469000..."
3,18,247,EL DONCELLO,18247,2018,1.106076e+09,1.791386,-75.193944,8809.0,0.0,...,7571.0,6287.0,1029.0,228.0,1095.0,171.0,3.154370,0.089867,18247,"MULTIPOLYGON (((4737450.122 1814755.048, 47374..."
4,18,256,EL PAUJÍL,18256,2018,1.234734e+09,1.617746,-75.234043,5795.0,0.0,...,6072.0,4066.0,639.0,108.0,916.0,99.0,3.529316,0.100309,18256,"MULTIPOLYGON (((4736905.653 1802381.382, 47376..."


> ## **1. Study Area and Project Setup**

>The study area is the municipality of Filandia, located in the department of Quindío, Colombia.

>The Colombian municipality vector layer is loaded with GeoPandas, reprojected to the official Colombian CRS EPSG:9377, and filtered to extract Filandia as the custom study area.

>The selected municipal geometry is subsequently converted to geographic coordinates for use in Google Earth Engine.

In [93]:
# define estudy area

# Load Colombian municipalities
gdf = gpd.read_file(root_folder / "municipios_colombia.gpkg")

# Reproject to the official Colombian CRS
gdf_9377 = gdf.to_crs(epsg=9377)

# Filter the municipality of Filandia
gdf_muni = gdf_9377[
    gdf_9377["MPIO_CNMBR"] == "FILANDIA"
].copy()

# Calculate municipality area in square kilometres
gdf_muni["area_km2"] = gdf_muni.geometry.area / 1_000_000

print("Selected municipality:", gdf_muni["MPIO_CNMBR"].iloc[0])
print("CRS:", gdf_muni.crs)
print(f"Area: {gdf_muni['area_km2'].iloc[0]:.2f} km²")

Selected municipality: FILANDIA
CRS: EPSG:9377
Area: 106.54 km²


In [100]:
# Convert the selected municipality to geographic coordinates
gdf_muni_4326 = gdf_muni.to_crs(epsg=4326)


### **1.1 Conversion of the Study Area to an Earth Engine Geometry**

The selected municipality is initially processed in EPSG:9377, the official projected coordinate reference system used for spatial analysis in Colombia.

To use the municipal boundary in Google Earth Engine, the selected GeoDataFrame is transformed to EPSG:4326 and converted into an Earth Engine geometry. This geometry is used as the spatial extent for the subsequent Sentinel-2 and Landsat 8 analyses.

In [102]:
# Convert the GeoPandas geometry to an Earth Engine geometry
ee_muni_geom = ee.Geometry(
    gdf_muni_4326.geometry.iloc[0].__geo_interface__
)

centroid = ee_muni_geom.centroid()

print("Study area successfully converted to an Earth Engine geometry.")

Study area successfully converted to an Earth Engine geometry.


In [103]:
#check the study area
Map = geemap.Map(center=[4.6693, -75.6715], zoom=11)

Map.addLayer(
    ee_muni_geom,
    {"color": "red"},
    "Study Area - Filandia"
)

Map

Map(center=[4.6693, -75.6715], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topri…

>## **2. Sentinel-2 SAVI Analysis**

>Sentinel-2 Surface Reflectance imagery is used to calculate the Soil-Adjusted Vegetation Index (SAVI) over the study area.

>Following the Earth Engine workflow developed in the Introductory Exercise, Sentinel-2 imagery is filtered spatially and temporally, cloud-masked using the Scene Classification Layer (SCL), and combined using median compositing. The original NDVI workflow is adapted here to calculate SAVI and to generate annual composites for the 2016–2025 multi-temporal analysis.

>### **2.1 Sentinel-2 Cloud Masking using SCL**

>Following the cloud-masking procedure used in the Introductory Exercise, the Sentinel-2 Scene Classification Layer (SCL) is used to identify and remove cloud shadows, medium- and high-probability clouds, and cirrus pixels before annual compositing and SAVI calculation.

In [104]:
# Sentinel-2 cloud masking function adapted from the Introductory Exercise
def mask_s2_clouds_scl(image):
    """Masks clouds, cloud shadows, and cirrus using the SCL band."""
    
    scl = image.select("SCL")

    # Identify unwanted pixel classes
    cloud_shadows = scl.eq(3)
    clouds_medium = scl.eq(8)
    clouds_high = scl.eq(9)
    cirrus = scl.eq(10)

    # Combine all mask conditions
    mask = (
        cloud_shadows
        .Or(clouds_medium)
        .Or(clouds_high)
        .Or(cirrus)
        .Not()
    )

    # Apply the mask
    return image.updateMask(mask)

print("Sentinel-2 SCL cloud masking function defined successfully.")

Sentinel-2 SCL cloud masking function defined successfully.


>### **2.2 Annual Sentinel-2 SAVI Composite**

>Sentinel-2 Surface Reflectance imagery is filtered by study area, year, and a maximum cloud percentage of 30%. The SCL cloud mask is applied to each image, and a pixel-wise annual median composite is generated.

>The Soil-Adjusted Vegetation Index (SAVI) is calculated using the red (B4) and near-infrared (B8) bands, with a soil adjustment factor of L = 0.5. The resulting SAVI image is clipped to the municipal boundary.

In [105]:
# SAVI soil adjustment factor
L = 0.5

print(f"SAVI soil adjustment factor (L): {L}")

SAVI soil adjustment factor (L): 0.5


In [114]:
# Compute an annual Sentinel-2 SAVI composite
def compute_annual_savi_s2(year):
    
    date_start = ee.Date.fromYMD(year, 1, 1)
    date_end = ee.Date.fromYMD(year + 1, 1, 1)

    # Filter Sentinel-2 collection
    s2_collection = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(ee_muni_geom)
        .filterDate(date_start, date_end)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 60))
        .map(mask_s2_clouds_scl)
    )

    # Compute annual median composite
    median_image = s2_collection.median()

    # Convert Sentinel-2 bands to surface reflectance
    red = median_image.select("B4").divide(10000)
    nir = median_image.select("B8").divide(10000)

    # Calculate SAVI and clip to the municipality boundary
    savi_image = (
        nir.subtract(red)
        .divide(nir.add(red).add(L))
        .multiply(1 + L)
        .rename("SAVI")
        .clip(ee_muni_geom)
    )

    return savi_image.set("year", year)

print("Annual Sentinel-2 SAVI function defined successfully.")

Annual Sentinel-2 SAVI function defined successfully.


>### **2.3 Sentinel-2 Data Availability**

>Sentinel-2 image availability is checked for each year before generating the annual SAVI time series. This step confirms whether sufficient observations are available within the study area for the selected period.

In [115]:
# Check Sentinel-2 image availability by year

s2_years = list(range(2017, 2026))

for year in s2_years:
    date_start = ee.Date.fromYMD(year, 1, 1)
    date_end = ee.Date.fromYMD(year + 1, 1, 1)

    annual_count = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(ee_muni_geom)
        .filterDate(date_start, date_end)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 60))
        .size()
        .getInfo()
    )

    print(f"{year}: {annual_count}")

2017: 1
2018: 4
2019: 24
2020: 31
2021: 21
2022: 18
2023: 16
2024: 33
2025: 26


>### **2.4 Annual Sentinel-2 SAVI Time Series**

>Annual Sentinel-2 SAVI composites are generated for each year with available Surface Reflectance observations. For each year, cloud-masked imagery is combined using a pixel-wise median composite, and SAVI is calculated from the red and near-infrared bands.

In [116]:
# Generate annual Sentinel-2 SAVI composites from 2017 to 2025

s2_years = list(range(2017, 2026))

s2_annual_savi = ee.ImageCollection.fromImages(
    [compute_annual_savi_s2(year) for year in s2_years]
)

print("Annual Sentinel-2 SAVI collection created.")
print("Number of annual images:", s2_annual_savi.size().getInfo())

Annual Sentinel-2 SAVI collection created.
Number of annual images: 9


>### **2.5 Sentinel-2 SAVI Visualization**

>The annual Sentinel-2 SAVI composite for 2025 is visualized to evaluate the spatial distribution of vegetation conditions within the study area.

>A color scale is applied to distinguish lower and higher SAVI values, and the municipal boundary is included as a spatial reference.

In [118]:
# Select Sentinel-2 SAVI composite for 2025
savi_2025_s2 = s2_annual_savi.filter(
    ee.Filter.eq("year", 2025)
).first()

# SAVI visualization parameters based on the 2nd and 98th percentiles
savi_vis = {
    "min": 0.24,
    "max": 0.64,
    "palette": [
        "#8C510A",  # Brown - lower SAVI
        "#D8B365",  # Light brown / ochre
        "#FFF176",  # Yellow
        "#A6D96A",  # Light green
        "#4DAC26",  # Medium green
        "#006837"   # Dark green - higher SAVI
    ]
}

# Create interactive map
Map_s2 = geemap.Map(center=[4.6693, -75.6715], zoom=11)

# Add SAVI layer
Map_s2.addLayer(
    savi_2025_s2,
    savi_vis,
    "Sentinel-2 SAVI 2025"
)

# Add municipality boundary
Map_s2.addLayer(
    ee_muni_geom,
    {"color": "red"},
    "Study Area"
)

Map_s2

Map(center=[4.6693, -75.6715], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topri…

>### **2.6 Sentinel-2 SAVI for 2016**

>Sentinel-2 Surface Reflectance imagery is not available for the complete 2016 period in the selected Earth Engine collection. Therefore, Sentinel-2 Level-1C imagery is used for 2016.

>Clouds and cirrus are masked using the QA60 band, and SAVI is calculated from the red (B4) and near-infrared (B8) bands. The resulting image is used to complete the 2016–2025 Sentinel-2 time series.

In [119]:
# Cloud masking function for Sentinel-2 Level-1C imagery in 2016
def mask_s2_2016(image):
    """Masks clouds and cirrus using the QA60 band."""

    qa = image.select("QA60")

    # QA60 bit 10 = opaque clouds
    # QA60 bit 11 = cirrus clouds
    cloud_mask = (
        qa.bitwiseAnd(1 << 10).eq(0)
        .And(qa.bitwiseAnd(1 << 11).eq(0))
    )

    return image.updateMask(cloud_mask)

In [120]:
# Generate Sentinel-2 SAVI composite for 2016

s2_2016_collection = (
    ee.ImageCollection("COPERNICUS/S2_HARMONIZED")
    .filterBounds(ee_muni_geom)
    .filterDate("2016-01-01", "2017-01-01")
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 30))
    .map(mask_s2_2016)
)

print(
    "Sentinel-2 images available for 2016:",
    s2_2016_collection.size().getInfo()
)

# Compute annual median composite
median_2016 = s2_2016_collection.median()

# Convert bands to reflectance
red_2016 = median_2016.select("B4").divide(10000)
nir_2016 = median_2016.select("B8").divide(10000)

# Calculate SAVI
savi_2016_s2 = (
    nir_2016.subtract(red_2016)
    .divide(nir_2016.add(red_2016).add(L))
    .multiply(1 + L)
    .rename("SAVI")
    .clip(ee_muni_geom)
    .set("year", 2016)
)

print("Sentinel-2 SAVI composite for 2016 created successfully.")

Sentinel-2 images available for 2016: 4
Sentinel-2 SAVI composite for 2016 created successfully.


In [121]:
# Combine 2016 with the annual Sentinel-2 SAVI collection from 2017 to 2025

s2_savi_images = [savi_2016_s2]

for year in range(2017, 2026):
    annual_image = (
        s2_annual_savi
        .filter(ee.Filter.eq("year", year))
        .first()
    )
    
    s2_savi_images.append(annual_image)

# Create the complete annual SAVI ImageCollection
s2_complete_savi = (
    ee.ImageCollection.fromImages(s2_savi_images)
    .sort("year")
)

print("Complete Sentinel-2 SAVI time series created.")
print("Number of annual images:", s2_complete_savi.size().getInfo())

Complete Sentinel-2 SAVI time series created.
Number of annual images: 10


In [122]:
# Verify the years included in the Sentinel-2 SAVI time series

s2_year_list = s2_complete_savi.aggregate_array("year").getInfo()

print("Years included:")
print(s2_year_list)

Years included:
[2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]


>### **2.7 Interactive Multi-Temporal Sentinel-2 SAVI Visualization**

>The annual Sentinel-2 SAVI composites from 2016 to 2025 are displayed in a single interactive map.

>Each year is added as an independent layer using the same visualization range and color palette, allowing consistent visual comparison of SAVI spatial patterns through time.

In [123]:
# Visualization parameters applied consistently to all years
savi_vis = {
    "min": 0.24,
    "max": 0.64,
    "palette": [
        "#8C510A",  # Brown - lower SAVI
        "#D8B365",  # Ochre
        "#FFF176",  # Yellow
        "#A6D96A",  # Light green
        "#4DAC26",  # Medium green
        "#006837"   # Dark green - higher SAVI
    ]
}

# Create interactive map
Map_s2_multi = geemap.Map(
    center=[4.6693, -75.6715],
    zoom=11
)

# Add one SAVI layer for each year
for year in range(2016, 2026):
    
    annual_savi = (
        s2_complete_savi
        .filter(ee.Filter.eq("year", year))
        .first()
    )

    Map_s2_multi.addLayer(
        annual_savi,
        savi_vis,
        f"Sentinel-2 SAVI {year}",
        False
    )

# Add municipality boundary
Map_s2_multi.addLayer(
    ee_muni_geom,
    {"color": "red"},
    "Study Area",
    True
)

Map_s2_multi

Map(center=[4.6693, -75.6715], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topri…

>### **2.9 Multi-Temporal Sentinel-2 SAVI Stack**

>The annual Sentinel-2 SAVI composites from 2016 to 2025 are combined into a single ten-band image stack.

>Each band represents one year of the time series and is renamed sequentially from `SAVI_2016` to `SAVI_2025`.

In [140]:
# Create the 10-band Sentinel-2 SAVI stack

s2_savi_stack = (
    s2_complete_savi
    .sort("year")
    .toBands()
    .rename([f"SAVI_{year}" for year in range(2016, 2026)])
)

print("Sentinel-2 SAVI multi-temporal stack created successfully.")

Sentinel-2 SAVI multi-temporal stack created successfully.


In [141]:
# Verify the Sentinel-2 SAVI stack

s2_band_names = s2_savi_stack.bandNames().getInfo()

print("Sentinel-2 SAVI stack bands:")
print(s2_band_names)
print("Number of bands:", len(s2_band_names))

Sentinel-2 SAVI stack bands:
['SAVI_2016', 'SAVI_2017', 'SAVI_2018', 'SAVI_2019', 'SAVI_2020', 'SAVI_2021', 'SAVI_2022', 'SAVI_2023', 'SAVI_2024', 'SAVI_2025']
Number of bands: 10


>## **3. Landsat 8 SAVI Analysis**

>Landsat 8 Collection 2 Level-2 Surface Reflectance imagery is used to calculate annual SAVI composites over the study area from 2016 to 2025.

>Clouds and other invalid pixels are masked using the `QA_PIXEL` band. Surface Reflectance scaling factors are then applied to the red and near-infrared bands before calculating SAVI.

>### **3.1 Landsat 8 Cloud Masking and Surface Reflectance Scaling**

>The `QA_PIXEL` quality band is used to remove fill pixels, dilated clouds, cirrus, clouds, cloud shadows, and snow.

>The Collection 2 Level-2 Surface Reflectance scale factor and offset are applied to the optical bands before SAVI calculation.

In [142]:
# Landsat 8 cloud masking and Surface Reflectance scaling function

def mask_l8_clouds(image):
    """Masks invalid pixels using QA_PIXEL and applies SR scaling."""

    qa = image.select("QA_PIXEL")

    # Mask fill, dilated cloud, cirrus, cloud, cloud shadow, and snow
    mask = (
        qa.bitwiseAnd(1 << 0).eq(0)
        .And(qa.bitwiseAnd(1 << 1).eq(0))
        .And(qa.bitwiseAnd(1 << 2).eq(0))
        .And(qa.bitwiseAnd(1 << 3).eq(0))
        .And(qa.bitwiseAnd(1 << 4).eq(0))
        .And(qa.bitwiseAnd(1 << 5).eq(0))
    )

    # Apply mask and Surface Reflectance scaling
    scaled_bands = (
        image
        .updateMask(mask)
        .select(["SR_B4", "SR_B5"])
        .multiply(0.0000275)
        .add(-0.2)
    )

    return scaled_bands

print("Landsat 8 cloud masking and scaling function defined successfully.")

Landsat 8 cloud masking and scaling function defined successfully.


>### **3.2 Annual Landsat 8 SAVI Composite**

>Landsat 8 Surface Reflectance imagery is filtered by study area and year. After applying the QA_PIXEL cloud mask and Surface Reflectance scaling factors, a pixel-wise annual median composite is generated.

>SAVI is calculated from the red (SR_B4) and near-infrared (SR_B5) bands using a soil adjustment factor of L = 0.5.

In [143]:
# Compute an annual Landsat 8 SAVI composite

def compute_annual_savi_l8(year):
    
    date_start = ee.Date.fromYMD(year, 1, 1)
    date_end = ee.Date.fromYMD(year + 1, 1, 1)

    # Filter Landsat 8 collection
    l8_collection = (
        ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
        .filterBounds(ee_muni_geom)
        .filterDate(date_start, date_end)
        .map(mask_l8_clouds)
    )

    # Compute annual median composite
    median_image = l8_collection.median()

    # Select red and near-infrared bands
    red = median_image.select("SR_B4")
    nir = median_image.select("SR_B5")

    # Calculate SAVI
    savi_image = (
        nir.subtract(red)
        .divide(nir.add(red).add(L))
        .multiply(1 + L)
        .rename("SAVI")
        .clip(ee_muni_geom)
    )

    return savi_image.set("year", year)

print("Annual Landsat 8 SAVI function defined successfully.")

Annual Landsat 8 SAVI function defined successfully.


>### **3.3 Landsat 8 Data Availability**

>Landsat 8 image availability is evaluated for each year from 2016 to 2025 before generating the annual SAVI composites. This step verifies that sufficient observations are available within the study area for the complete analysis period.

In [144]:
# Check Landsat 8 image availability by year

l8_years = list(range(2016, 2026))

for year in l8_years:
    date_start = ee.Date.fromYMD(year, 1, 1)
    date_end = ee.Date.fromYMD(year + 1, 1, 1)

    annual_count = (
        ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
        .filterBounds(ee_muni_geom)
        .filterDate(date_start, date_end)
        .size()
        .getInfo()
    )

    print(f"{year}: {annual_count}")

2016: 21
2017: 18
2018: 18
2019: 20
2020: 18
2021: 19
2022: 19
2023: 20
2024: 18
2025: 21


>### **3.4 Annual Landsat 8 SAVI Time Series**

>Annual Landsat 8 SAVI composites are generated for each year from 2016 to 2025.

>For each year, cloud-masked and scaled Surface Reflectance imagery is combined using a pixel-wise median composite, and SAVI is calculated from the red and near-infrared bands.

In [145]:
# Generate annual Landsat 8 SAVI composites from 2016 to 2025

l8_years = list(range(2016, 2026))

l8_annual_savi = ee.ImageCollection.fromImages(
    [compute_annual_savi_l8(year) for year in l8_years]
)

print("Annual Landsat 8 SAVI collection created.")
print("Number of annual images:", l8_annual_savi.size().getInfo())

Annual Landsat 8 SAVI collection created.
Number of annual images: 10


In [146]:
# Verify the years included in the Landsat 8 SAVI time series

l8_year_list = l8_annual_savi.aggregate_array("year").getInfo()

print("Years included:")
print(l8_year_list)

Years included:
[2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]


>### **3.5 Interactive Multi-Temporal Landsat 8 SAVI Visualization**

>The annual Landsat 8 SAVI composites from 2016 to 2025 are displayed in a single interactive map.

>Each year is added as an independent layer using the same visualization range and color palette to allow consistent comparison of SAVI spatial patterns through time.

In [131]:
# Visualization parameters applied consistently to all years
savi_vis_l8 = {
    "min": 0.24,
    "max": 0.64,
    "palette": [
        "#8C510A",  # Brown - lower SAVI
        "#D8B365",  # Ochre
        "#FFF176",  # Yellow
        "#A6D96A",  # Light green
        "#4DAC26",  # Medium green
        "#006837"   # Dark green - higher SAVI
    ]
}

# Create interactive map
Map_l8_multi = geemap.Map(
    center=[4.6693, -75.6715],
    zoom=11
)

# Add one SAVI layer for each year
for year in range(2016, 2026):

    annual_savi = (
        l8_annual_savi
        .filter(ee.Filter.eq("year", year))
        .first()
    )

    Map_l8_multi.addLayer(
        annual_savi,
        savi_vis_l8,
        f"Landsat 8 SAVI {year}",
        False
    )

# Add municipality boundary
Map_l8_multi.addLayer(
    ee_muni_geom,
    {"color": "red"},
    "Study Area",
    True
)

Map_l8_multi

Map(center=[4.6693, -75.6715], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topri…

>### **3.6 Multi-Temporal Landsat 8 SAVI Stack**

>The annual Landsat 8 SAVI composites from 2016 to 2025 are combined into a single ten-band image stack.

>Each band represents one year of the time series and is renamed sequentially from `SAVI_2016` to `SAVI_2025`.

In [147]:
# Create the 10-band Landsat 8 SAVI stack

l8_savi_stack = (
    l8_annual_savi
    .sort("year")
    .toBands()
    .rename([f"SAVI_{year}" for year in range(2016, 2026)])
)

print("Landsat 8 SAVI multi-temporal stack created successfully.")

Landsat 8 SAVI multi-temporal stack created successfully.


In [148]:
# Verify the Landsat 8 SAVI stack

l8_band_names = l8_savi_stack.bandNames().getInfo()

print("Landsat 8 SAVI stack bands:")
print(l8_band_names)
print("Number of bands:", len(l8_band_names))

Landsat 8 SAVI stack bands:
['SAVI_2016', 'SAVI_2017', 'SAVI_2018', 'SAVI_2019', 'SAVI_2020', 'SAVI_2021', 'SAVI_2022', 'SAVI_2023', 'SAVI_2024', 'SAVI_2025']
Number of bands: 10


>## **4. Export of Multi-Temporal SAVI Stacks**

>The final Sentinel-2 and Landsat 8 SAVI stacks are exported to Google Drive as multi-band GeoTIFF files.

>Both raster stacks contain ten annual SAVI bands corresponding to the 2016–2025 period and are exported using the Colombian projected coordinate reference system MAGNA-SIRGAS 2018 / Origen-Nacional (EPSG:9377).

In [149]:
# EPSG:9377 - MAGNA-SIRGAS 2018 / Origen-Nacional
epsg_9377_wkt = """
PROJCS["MAGNA-SIRGAS 2018 / Origen-Nacional",
    GEOGCS["MAGNA-SIRGAS 2018",
        DATUM["Marco_Geocentrico_Nacional_de_Referencia",
            SPHEROID["GRS 1980",6378137,298.257222101]],
        PRIMEM["Greenwich",0],
        UNIT["degree",0.0174532925199433]],
    PROJECTION["Transverse_Mercator"],
    PARAMETER["latitude_of_origin",4],
    PARAMETER["central_meridian",-73],
    PARAMETER["scale_factor",0.9992],
    PARAMETER["false_easting",5000000],
    PARAMETER["false_northing",2000000],
    UNIT["metre",1]]
"""

# Verify that Earth Engine recognizes the projection
projection_9377 = ee.Projection(epsg_9377_wkt)

print("Export projection:")
print(projection_9377.getInfo())

Export projection:
{'type': 'Projection', 'wkt': 'PROJCS["MAGNA-SIRGAS 2018 / Origen-Nacional", \n  GEOGCS["MAGNA-SIRGAS 2018", \n    DATUM["Marco_Geocentrico_Nacional_de_Referencia", \n      SPHEROID["GRS 1980", 6378137.0, 298.257222101]], \n    PRIMEM["Greenwich", 0.0], \n    UNIT["degree", 0.017453292519943295], \n    AXIS["Longitude", EAST], \n    AXIS["Latitude", NORTH]], \n  PROJECTION["Transverse_Mercator"], \n  PARAMETER["central_meridian", -73.0], \n  PARAMETER["latitude_of_origin", 4.0], \n  PARAMETER["scale_factor", 0.9992], \n  PARAMETER["false_easting", 5000000.0], \n  PARAMETER["false_northing", 2000000.0], \n  UNIT["m", 1.0], \n  AXIS["x", EAST], \n  AXIS["y", NORTH]]', 'transform': [1, 0, 0, 0, 1, 0]}


>### **4.1 Sentinel-2 SAVI Stack Export**

>The Sentinel-2 SAVI stack is exported as a ten-band GeoTIFF at 10 m spatial resolution using EPSG:9377.

In [150]:
# Export Sentinel-2 10-band SAVI stack to Google Drive

task_s2 = ee.batch.Export.image.toDrive(
    image=s2_savi_stack,
    description="Sentinel2_SAVI_Stack_2016_2025",
    folder="SAVI_MultiSensor_Exports",
    fileNamePrefix="Sentinel2_SAVI_Stack_2016_2025",
    region=ee_muni_geom,
    scale=10,
    crs=epsg_9377_wkt,
    maxPixels=1e13,
    fileFormat="GeoTIFF",
    formatOptions={
        "cloudOptimized": True
    }
)

task_s2.start()

print("Sentinel-2 export task started.")
print("Task status:", task_s2.status()["state"])

Sentinel-2 export task started.
Task status: READY


>### **4.2 Landsat 8 SAVI Stack Export**

>The Landsat 8 SAVI stack is exported as a ten-band GeoTIFF at 30 m spatial resolution using EPSG:9377.

In [151]:
# Export Landsat 8 10-band SAVI stack to Google Drive

task_l8 = ee.batch.Export.image.toDrive(
    image=l8_savi_stack,
    description="Landsat8_SAVI_Stack_2016_2025",
    folder="SAVI_MultiSensor_Exports",
    fileNamePrefix="Landsat8_SAVI_Stack_2016_2025",
    region=ee_muni_geom,
    scale=30,
    crs=epsg_9377_wkt,
    maxPixels=1e13,
    fileFormat="GeoTIFF",
    formatOptions={
        "cloudOptimized": True
    }
)

task_l8.start()

print("Landsat 8 export task started.")
print("Task status:", task_l8.status()["state"])

Landsat 8 export task started.
Task status: READY


>### **4.3 Export Task Verification**

>The export status of both multi-temporal SAVI stacks is checked to confirm that the GeoTIFF files were successfully generated in Google Drive.

In [160]:
# Summarize export results

s2_status = task_s2.status()
l8_status = task_l8.status()

print("Sentinel-2:", s2_status["state"])
print("Landsat 8:", l8_status["state"])

if s2_status["state"] == "COMPLETED" and l8_status["state"] == "COMPLETED":
    print("\nBoth SAVI stacks were exported successfully.")
else:
    print("\nAt least one export is still running or requires review.")

Sentinel-2: COMPLETED
Landsat 8: COMPLETED

Both SAVI stacks were exported successfully.


In [161]:
print("Sentinel-2:", task_s2.status()["state"])
print("Landsat 8:", task_l8.status()["state"])

Sentinel-2: COMPLETED
Landsat 8: COMPLETED


In [162]:
# Verify the exported GeoTIFF files

sentinel_path = root_folder / "Sentinel2_SAVI_Stack_2016_2025.tif"
landsat_path = root_folder / "Landsat8_SAVI_Stack_2016_2025.tif"

def inspect_raster(raster_path, raster_name):
    with rasterio.open(raster_path) as src:
        print(f"\n{raster_name}")
        print("-" * 50)
        print("CRS:", src.crs)
        print("Number of bands:", src.count)
        print("Resolution:", src.res)
        print("Data type:", src.dtypes[0])

inspect_raster(sentinel_path, "Sentinel-2 SAVI Stack")
inspect_raster(landsat_path, "Landsat 8 SAVI Stack")


Sentinel-2 SAVI Stack
--------------------------------------------------
CRS: PROJCS["MAGNA-SIRGAS 2018 / Origen-Nacional",GEOGCS["MAGNA-SIRGAS 2018",DATUM["Marco_Geocentrico_Nacional_de_Referencia",SPHEROID["GRS 1980",6378137,298.257222101004,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6686"]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",4],PARAMETER["central_meridian",-73],PARAMETER["scale_factor",0.9992],PARAMETER["false_easting",5000000],PARAMETER["false_northing",2000000],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]
Number of bands: 10
Resolution: (10.0, 10.0)
Data type: float64

Landsat 8 SAVI Stack
--------------------------------------------------
CRS: PROJCS["MAGNA-SIRGAS 2018 / Origen-Nacional",GEOGCS["MAGNA-SIRGAS 2018",DATUM["Marco_Geocentrico_Nacional_de_Referencia",SPHEROID["GRS 1980",6378137,298.257222101004,AUTHORITY["EPSG","701